# SAMueL Create k-fold Data Sets

## Plain English summary
Create and save the data in 5 kfold splits.

## Load imports

In [1]:
import pandas as pd
import numpy as np
import yaml

from dataclasses import dataclass
from sklearn.model_selection import train_test_split

# Turn warnings off to keep notebook tidy
import warnings
warnings.filterwarnings("ignore")

## Set up paths and filenames

In [2]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    data_read_path: str = './stroke_utilities/data/'
    data_read_filename: str = 'reformatted_data_thrombolysis_decision.csv'
    data_save_path: str = './stroke_utilities/data'
    notebook: str = ''

paths = Paths()

# Load data



In [3]:
filename = paths.data_read_path + paths.data_read_filename
data = pd.read_csv(filename)


Ensure all values are float and shuffle

In [4]:
data = data.sample(frac=1.0, random_state=42)

## Limit to scan with enough time for thrombolysis

In [5]:
from stroke_utilities.scenario import create_masks

In [6]:
with open('./stroke_utilities/fixed_params.yml') as f:
    fixed_params = yaml.safe_load(f)

In [7]:
# allowed_onset_to_needle_time_mins = fixed_params['allowed_onset_to_needle_time_mins']
# minutes_left = fixed_params['minutes_left']
allowed_onset_to_scan_time = fixed_params['allowed_onset_to_scan_time']

In [8]:
def restrict_to_onset_to_scan_on_time(big_data):    
    # Time left after scan for thrombolysis
    big_data['onset_to_scan_time'] = (
        big_data['onset_to_arrival_time'] + 
        big_data['arrival_to_scan_time']
        )

    mask_to_include = big_data['onset_to_scan_time'] <= allowed_onset_to_scan_time

    # Restrict the data to these patients:
    big_data = big_data[mask_to_include]
    return big_data

In [9]:
data = restrict_to_onset_to_scan_on_time(data)

In [10]:
np.nan <= 240

False

In [11]:
# mask = data['onset_to_arrival_time'] <= 240
# data = data[mask]

## Limit to 10 features and thrombolysis label

In [12]:
features_to_use = [
    'stroke_team_id',
    'stroke_severity',
    'prior_disability',
    'age',
    'infarction',
    'onset_to_arrival_time',
    'precise_onset_known',
    'onset_during_sleep',
    'arrival_to_scan_time',
    'afib_anticoagulant',
    'year',    
    'thrombolysis'
]

## Create stratification based on hospital and thrombolysis use

In [13]:
strat = data['stroke_team_id'].map(str) + '-' + data['thrombolysis'].map(str)

## Create and save 10k test and train sets

In [13]:
make_new_split = False

if make_new_split:
    # Split X and y
    X = data[features_to_use].drop('thrombolysis', axis=1)
    y = data['thrombolysis']
    
    # Create train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=10000, stratify=strat, random_state=42)
    train = pd.concat([X_train, y_train], axis=1)
    test = pd.concat([X_test, y_test], axis=1)
    
    # Save
    train.to_csv(f'{paths.data_save_path}/cohort_10000_train.csv', index=False)
    test.to_csv(f'{paths.data_save_path}/cohort_10000_test.csv', index=False)

## Bodge missing data back in

Should have included some extra columns above.

Now find out which patient in the test and train data matches each patient from the original data so that we can add in those extra columns.

First check that all rows of data are unique:

In [12]:
train = pd.read_csv(f'{paths.data_save_path}/cohort_10000_train.csv')
test = pd.read_csv(f'{paths.data_save_path}/cohort_10000_test.csv')

In [15]:
for d in [data, train, test]:
    print(len(d) == len(d[['stroke_team_id'] + features_to_use].drop_duplicates()))

False
False
True


In [14]:
thrombolysis_outcome_fields = [
    'prior_disability',
    'stroke_severity',
    'stroke_team',
    'onset_to_thrombolysis',
    'age',
    'precise_onset_known',
    'any_afib_diagnosis',
    'discharge_disability',
    'thrombectomy'
]

extra_cols = [c for c in thrombolysis_outcome_fields if c not in features_to_use]
all_cols = features_to_use + extra_cols

Create new data for the outcome model:

In [17]:
data['onset_to_thrombolysis'] = data['onset_to_arrival_time'] + data['arrival_to_scan_time']

In [18]:
data['any_afib_diagnosis'] = data['atrial_fibrillation']

Pick out duplicate data in the prediction features and check whether they're also duplicates for the outcome features.

In [19]:
d_dup = data[data[features_to_use].duplicated(keep=False)]
print(f'{len(d_dup)} repeats')
display(d_dup[all_cols].sort_values(all_cols).T)

26 repeats


,201919,232215,167394,186712,162279,107754,156143,163442,138168,138043,...,112438,145839,174233,178892,6795,3139,140542,149365,228231,235770
stroke_team_id,11,11,11,11,84,84,84,84,84,84,...,84,84,84,84,84,84,84,84,110,110
stroke_severity,1,1,12,12,3,3,3,3,8,8,...,19,19,19,19,23,23,27,27,11,11
prior_disability,1,1,0,0,0,0,1,1,0,0,...,0,0,4,4,1,1,5,5,0,0
age,77.5,77.5,82.5,82.5,77.5,77.5,92.5,92.5,82.5,82.5,...,62.5,62.5,92.5,92.5,87.5,87.5,92.5,92.5,62.5,62.5
infarction,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
onset_to_arrival_time,75.0,75.0,36.0,36.0,32.0,32.0,90.0,90.0,89.0,89.0,...,65.0,65.0,61.0,61.0,42.0,42.0,134.0,134.0,20.0,20.0
precise_onset_known,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,0,0
onset_during_sleep,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
arrival_to_scan_time,42.0,42.0,56.0,56.0,16.0,16.0,61.0,61.0,8.0,8.0,...,26.0,26.0,59.0,59.0,21.0,21.0,22.0,22.0,20.0,20.0
afib_anticoagulant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0


Any patients with duplicated prediction data also have the same data in the extra outcome columns. So use whichever entry.

Loop over each patient in turn and find where their data is in the training and test data.

In [72]:
def find_patient_ids(df, data):
    ids = []
    for i in df.index:
        df_here = df.loc[i]
        masks = []
        for col in df.columns:
            if np.isnan(df_here[col]):
                mask_col = data[col].isna()
            else:
                mask_col = (data[col] == df_here[col])
            masks.append(mask_col)
        mask = pd.concat(masks, axis='columns').all(axis='columns')
        id = data[mask]['id'].values[0]
        ids.append(id)
    return ids

In [81]:
ids_train = find_patient_ids(train, data)
ids_test = find_patient_ids(test, data)

In [86]:
def make_full_data(ids, df, data, extra_cols):
    series_ids = pd.Series(ids)
    series_ids.name = 'id'
    
    # Place patient IDs in this data:
    df_here = pd.concat((df, series_ids), axis='columns')
    
    # Copy over more data from original file:
    df_here = pd.merge(df_here, data[['id'] + extra_cols], on='id', how='left')
    return df_here

In [87]:
train_full = make_full_data(ids_train, train, data, extra_cols)
test_full = make_full_data(ids_test, test, data, extra_cols)

In [90]:
train_full = train_full.drop('stroke_team', axis='columns')
test_full = test_full.drop('stroke_team', axis='columns')

## Make data for outcomes

Remove patients who received thrombectomy and have missing discharge disability.

In [48]:
train_full = pd.read_csv(f'{paths.data_save_path}/cohort_10000_train_full.csv')
test_full = pd.read_csv(f'{paths.data_save_path}/cohort_10000_test_full.csv')

In [55]:
def clean_data_for_outcomes(df):
    print(f'Patients at start:                  {len(df):6.0f}')
    mask = (df['thrombectomy'] < 1)
    df = df.loc[mask]
    print(f'Patients without thrombectomy:      {len(df):6.0f}')
    mask = df['discharge_disability'].notna()
    df = df.loc[mask]
    print(f'Patients with discharge disability: {len(df):6.0f}')
    mask = (df['infarction'] > 0)
    df = df.loc[mask]
    print(f'Patients with infarction:           {len(df):6.0f}')
    return df.copy()

In [56]:
train_outcomes = clean_data_for_outcomes(train_full)

Patients at start:                  104213
Patients without thrombectomy:      101145
Patients with discharge disability: 100402
Patients with infarction:            85065


In [57]:
test_outcomes = clean_data_for_outcomes(test_full)

Patients at start:                   10000
Patients without thrombectomy:        9694
Patients with discharge disability:   9620
Patients with infarction:             8121


In [52]:
cols_to_keep = ['stroke_team_id'] + [t for t in thrombolysis_outcome_fields]
cols_to_keep.remove('thrombectomy')
cols_to_keep.remove('stroke_team')

train_outcomes = train_outcomes[cols_to_keep]
test_outcomes = test_outcomes[cols_to_keep]

In [53]:
# Save
train_outcomes.to_csv(f'{paths.data_save_path}/cohort_10000_train_outcomes.csv', index=False)
test_outcomes.to_csv(f'{paths.data_save_path}/cohort_10000_test_outcomes.csv', index=False)